# Module 2 - AI-Assisted Development in Practice

This companion notebook is the durable student reference for Module 2.

Core principle:

> AI assistance is valuable when it is embedded in a disciplined developer loop.

Module 2 is not a tool tour. It is about framing tasks, planning before code, writing bounded prompts, verifying generated work, and keeping human ownership of code quality.

## How to use this notebook

- Use the slide deck for the live teaching spine.
- Use this notebook for examples, prompt patterns, exercises, worked solutions, and review checklists.
- Use `work_items_api_starter/` for hands-on work.
- Use `work_items_api_solution/` only after trying the lab or when a live demo needs a fallback.

Sections marked **Optional / skim if needed** can be skipped when a cohort needs more time for labs.

Important solution locations:

| Material | Where to look |
|---|---|
| Lab 1 solution notes | Section 8 in this notebook and `work_items_api_solution/` |
| Lab 2 full worked solution | Section 14.1–14.3 in this notebook and `work_items_api_solution/tests/test_legacy_work_item_logic.py` |
| Homework solutions | `module2_homework.md` |
| Demo prompts and fallbacks | `module2_demo_runbook.md` |

## Solution index and synchronization notes

This notebook deliberately contains both **student-facing exercises** and **instructor/solution material**. In live class, reveal solution sections only after students attempt the relevant task.

| Topic | Student work | Solution / model answer |
|---|---|---|
| Prompt rescue | Section 4 | Section 4 `Solution` block |
| Lab 1 CRUD API | Section 7 | Section 8 and `work_items_api_solution/` |
| Test-the-tests exercise | Section 9 | `review_exercises/corrected_generated_tests.py` |
| Lab 2 legacy code | Section 14 | Sections 14.1–14.3 and `work_items_api_solution/tests/test_legacy_work_item_logic.py` |
| API client review | Section 12 | `work_items_api_solution/app/client.py` and `tests/test_api_client.py` |
| Homework | Section 17 | `module2_homework.md` |

The slide deck uses Section 14 as the Lab 2 marker. The detailed Lab 2 solution is intentionally placed immediately after the Lab 2 task description.

## Learning outcomes

By the end, you should be able to:

1. use Frame → Plan → Prompt → Execute → Verify → Reflect;
2. ask AI for plans before code;
3. use AI for implementation, debugging, refactoring, tests, comments, documentation, API clients, and review;
4. work safely in existing codebases with Map → Bound → Verify;
5. critique AI-generated tests and diffs;
6. prevent sensitive data leakage.

## 1. The workflow spine

| Step | Question | Student artifact |
|---|---|---|
| Frame | What are we trying to do? | task frame |
| Plan | What is the smallest safe approach? | implementation plan |
| Prompt | What bounded request should AI answer? | prompt with constraints |
| Execute | What changed? | code/test/docs diff |
| Verify | How do we know it works? | test output + manual checks |
| Reflect | What did we accept, reject, or learn? | prompt log / PR note |

A strong AI-assisted workflow keeps the human in control of every transition.

## 2. Worked example - planning before code

Weak prompt:

```text
Make the Work Items API work.
```

Stronger prompt:

```text
Do not write code yet.
Inspect this FastAPI Work Items API project and propose a minimal implementation plan for making the CRUD tests pass.
Include current behavior, files likely to change, files that should not change, missing repository behavior, steps, tests, verification commands, assumptions, and risks.
```

Why it is stronger:

- it prevents immediate broad edits;
- it makes scope visible;
- it asks for files out of scope;
- it ties the work to tests;
- it creates a reviewable intermediate artifact.

## 3. Annotated AI plan

A useful plan might say:

```text
Likely file to change: app/repository.py.
Do not change: app/main.py endpoint paths, app/models.py enums, tests unless a test is wrong.
Implementation: create/list first, then get/update/delete, then filters.
Risks: SQL injection if filters are interpolated; breaking response shape; weak tests that only check status codes.
Verification: pytest -q; python -m compileall app.
```

Human review comments:

- Good: it identifies the repository as the implementation boundary.
- Good: it keeps endpoint paths stable.
- Good: it names parameterized SQL as a risk.
- Watch: if it proposes pagination now, that is scope creep.
- Watch: if it proposes SQLAlchemy, that is unnecessary dependency churn for this lab.

## 4. Prompt layers for developers

Strong prompts usually include:

1. **Task** — what needs to happen.
2. **Context** — repo, file, ticket, stack trace, API contract.
3. **Constraints** — what must stay stable.
4. **Output format** — plan, diff, tests, checklist, docs.
5. **Verification** — tests, commands, review questions.

### 🫵 Exercise 1 - rewrite this weak prompt:

```text
Fix the API.
```

#### Solution

```text
Do not write code yet. The FastAPI Work Items API has failing CRUD acceptance tests. Inspect app/main.py, app/models.py, app/repository.py, and tests/test_work_items_api.py. Plan the smallest implementation in app/repository.py only. Keep endpoint paths and response shapes stable. Include tests to run, risks, assumptions, and verification commands.
```

## 5. Repository instructions as context engineering

A repository can guide AI through persistent context files and ordinary documentation.

Examples in the lab repos:

```text
.github/copilot-instructions.md
.github/prompts/plan-work-item-feature.prompt.md
.github/prompts/review-diff.prompt.md
docs/ai-working-agreement.md
docs/review-checklist.md
```

The deeper lesson:

> A repo that is legible to humans is usually more legible to AI.

## 6. Daily workflow coverage

AI can support many daily developer tasks:

| Activity | Good AI use | Human responsibility |
|---|---|---|
| Ticket clarification | identify missing acceptance criteria | decide requirements |
| Implementation | bounded edits | review diff |
| Debugging | hypotheses and evidence | confirm cause |
| Refactoring | behavior-preserving plan | ensure tests protect behavior |
| Tests | draft cases | inspect assertions |
| Comments/docs | draft actual behavior | remove noise and overclaims |
| API clients | scaffold client | review errors/timeouts/auth |
| PR review | checklist and summary | approve or reject |
| Changelog/release notes | summarize merged changes | product accuracy |

## 7. 🫵 Lab 1: vertical-slice implementation

Use:

```text
work_items_api_starter/
```

Run:

```bash
pytest -q
```

Expected: failures. The tests define the API behavior.

Implementation target:

```text
app/repository.py
```

Recommended slices:

1. create + list;
2. get;
3. patch;
4. delete;
5. filters;
6. review generated tests;
7. optional API client review.

## 8. Lab 1 solution notes

A correct repository implementation should:

- create a `WorkItem` with timestamps;
- persist enum values as strings;
- list items in deterministic order;
- build filters with parameterized SQL;
- return `None` for missing items;
- preserve unspecified fields in partial updates;
- refresh `updated_at` on updates;
- return `False` for deleting missing items.

A strong solution does **not**:

- rewrite endpoints unnecessarily;
- delete or weaken tests;
- add SQLAlchemy just to solve a small SQLite exercise;
- change response shapes;
- hide errors behind generic `200` responses.

The full solution is in:

```text
work_items_api_solution/
```

Key files:

```text
work_items_api_solution/app/repository.py
work_items_api_solution/tests/test_work_items_api.py
work_items_api_solution/app/client.py
work_items_api_solution/tests/test_api_client.py
```

### Solution walkthrough

The intended implementation boundary is the repository layer. `app/main.py` already defines the API contract and maps repository return values into HTTP behavior. Therefore, the safest implementation slice is to complete `app/repository.py` without changing endpoint paths or response shapes.

The important behaviors are:

1. **Create**: instantiate a `WorkItem`, insert it into SQLite, and return the created object.
2. **List**: return all rows ordered deterministically; apply optional status and priority filters using query parameters, not string interpolation.
3. **Get**: return a single item or `None` for a missing ID.
4. **Update**: load the existing item, apply only provided fields, preserve unspecified values, refresh `updated_at`, and persist.
5. **Delete**: delete by ID and return whether a row was actually deleted.

A good AI-assisted implementation prompt for the first slice is:

```text
Implement only the repository create/list behavior needed by the failing tests.
Keep endpoint paths and response shapes stable.
Use parameterized SQL.
Do not refactor app/main.py or app/models.py.
After editing, summarize the diff and list the exact tests to run.
```

A strong PR summary for the completed solution would say:

```text
Implemented SQLite-backed WorkItem repository behavior for create, list, get, patch, delete, and filters. Added/kept acceptance tests for payloads, deterministic listing, missing-item 404s, partial update preservation, deletion, status filtering, priority filtering, combined filtering, validation, and API-client behavior. Verified with `pytest -q` and `python -m compileall app`.
```

### Expected verification

From `work_items_api_solution/`:

```bash
pytest -q
python -m compileall app
```

Expected test result in the v4 sync pass:

```text
23 passed
```

## 9. Test the tests

Open:

```text
review_exercises/weak_generated_tests.py
```

The weak draft includes tests that look useful but do not prove enough.

Review questions:

- Does the test assert payload behavior?
- Does the filter test include non-matching data?
- Does the delete test prove the item is gone?
- Does the test name overclaim?

Compare with:

```text
review_exercises/corrected_generated_tests.py
```

Key lesson:

> A test name can lie. The assertion is where truth lives.

## 10. Debugging with AI

Bad prompt:

```text
Fix this failing test.
```

Better prompt:

```text
Do not patch yet.
Analyze this failing test and stack trace.
List the top 3 likely causes, evidence for and against each, the smallest check to confirm, and the first fix you would try.
```

Use AI to build hypotheses before accepting patches.

## 11. Refactoring and comments



Required compact topic.

### Safe refactoring pattern

1. Ask AI to explain current behavior.
2. Add or improve tests.
3. Ask for a behavior-preserving refactor plan.
4. Implement one small refactor.
5. Run tests.
6. Review diff for behavior changes.

### Comments/docstrings

Good comments explain intent, constraints, or surprising behavior.

Weak comments merely restate the code:

```python
# send request
response = client.get('/work-items')
```

Better docstring:

```python
"""List work items using server-side filters so callers do not fetch unnecessary data."""
```

## 12. API client generation and review

Required compact topic.

Prompt:

```text
Generate a small Python httpx client for the Work Items API.
Include create, list, get, update, and delete methods.
Use explicit timeouts.
Raise useful exceptions for non-2xx responses.
Add docstrings that explain intent.
Do not add authentication or retries yet.
```

Review checklist:

- Is there a timeout?
- Are errors raised clearly?
- Are response-shape assumptions visible?
- Are comments useful?
- Did AI invent auth headers or tokens?
- Can it be tested with `httpx.MockTransport`?

See:

```text
work_items_api_solution/app/client.py
work_items_api_solution/tests/test_api_client.py
```

### [Optional] Advanced API-client hardening

Hardening questions:

- Should errors have typed exceptions?
- Should retries exist, and for which status codes?
- Should the client support async usage?
- Where does authentication belong?
- What should be logged, and what must never be logged?
- Should the client expose raw responses or typed domain objects?

## 13. Existing codebases: Map → Bound → Verify

Existing codebases require more discipline than greenfield examples.

#### Map

```text
Produce a read-only architecture map. Do not suggest changes yet.
```

#### Bound

```text
Identify the smallest safe implementation slice. List files to touch and files to avoid.
```

#### Verify

```text
Review the diff for behavior changes, missing tests, backward compatibility, and manual checks.
```

Teaching shortcut:

> Explain → Test → Tiny fix.

## 14. Lab 2: legacy explanation, tests, and docs

Use:

```text
work_items_api_starter/lab2_assets/legacy_work_item_logic.py
```

This lab is required. The optional part is only the final refactor discussion.

Tasks:

1. Ask AI for a read-only explanation.
2. Challenge assumptions and overconfident claims.
3. Ask for edge cases.
4. Generate tests.
5. Review and improve the tests.
6. Draft documentation that matches actual behavior.
7. Optional / skim if needed: propose a behavior-preserving refactor plan without implementing it.

Prompt sequence:

```text
Explain this module for a new team member.
Focus on actual behavior and edge cases.
Do not suggest changes yet.
Base your explanation only on the visible code.
```

Then:

```text
Which of your claims are directly visible in the code, and which are assumptions?
```

Then:

```text
Write pytest tests for the most important behaviors.
The tests should prove behavior, not implementation details.
Use explicit assertions and cover aliases, missing due dates, done items, date parsing, and malformed date input.
```

The full worked solution follows in Sections 14.1–14.3. The executable solution tests are also in:

```text
work_items_api_solution/tests/test_legacy_work_item_logic.py
```

## 14.1 Lab 2 solution - behavior summary and edge cases

A strong AI-assisted explanation should be close to this model answer.

### Model explanation

`legacy_work_item_logic.py` contains two public helpers and one private parsing helper.

- `normalize_status(raw)` converts common partner status labels into canonical work-item statuses.
- `_parse_due(value)` converts `None`, empty strings, `datetime` values, and ISO-like strings into either `None` or a `datetime`.
- `is_overdue(item, now=None)` returns `True` only when the item is not `done`, has a due date, and that due date is earlier than the comparison time.

### Actual behavior to preserve

| Behavior | Evidence in code | Test implication |
|---|---|---|
| `None` status becomes `todo` | `if raw is None: return "todo"` | Test default status behavior. |
| Known aliases normalize | `_STATUS_ALIASES` dictionary | Test `new`, `WIP`, `completed`, etc. |
| Unknown statuses pass through | `.get(cleaned, cleaned)` | Test `Blocked` becomes `blocked`. |
| Done items are never overdue | early return in `is_overdue` | Test done + past due returns `False`. |
| Missing due date is not overdue | `_parse_due(...)` returns `None`; caller returns `False` | Test missing `due_at`/`due_date`. |
| Date-only strings are valid | `datetime.fromisoformat(value)` accepts `YYYY-MM-DD` | Test date-only interpretation. |
| Malformed date strings raise `ValueError` | `datetime.fromisoformat` raises | Test that behavior explicitly. |
| Unsupported due-date types raise `TypeError` | final branch in `_parse_due` | Optional extra test. |
| Naive/aware comparison is normalized one way | current time may be stripped of tzinfo if due date is naive | Mention as a caveat. |

### What AI commonly overclaims

A generated explanation may say this module has “robust date handling.” That is too strong. The code handles a few common forms, but it does not recover from malformed strings, validate time zones deeply, or provide user-friendly error messages.

A better phrasing is:

> The module accepts `datetime` values and ISO-like date strings. It treats missing due dates as not overdue, but malformed date strings raise exceptions and should be validated by callers if user-facing input is possible.

### Source code being explained

```python
from datetime import datetime, timezone
from typing import Any

_STATUS_ALIASES = {
    "new": "todo",
    "open": "todo",
    "todo": "todo",
    "wip": "in_progress",
    "doing": "in_progress",
    "in progress": "in_progress",
    "in_progress": "in_progress",
    "complete": "done",
    "completed": "done",
    "done": "done",
}


def normalize_status(raw: str | None) -> str:
    """Normalize common partner status labels while passing unknown values through."""
    if raw is None:
        return "todo"
    cleaned = raw.strip().lower().replace("-", "_")
    return _STATUS_ALIASES.get(cleaned, cleaned)


def _parse_due(value: Any) -> datetime | None:
    if value is None or value == "":
        return None
    if isinstance(value, datetime):
        return value
    if isinstance(value, str):
        return datetime.fromisoformat(value)
    raise TypeError(f"Unsupported due date value: {type(value).__name__}")


def is_overdue(item: dict[str, Any], *, now: datetime | None = None) -> bool:
    """Return True when an unfinished item has a due date earlier than now."""
    if normalize_status(item.get("status")) == "done":
        return False
    due = _parse_due(item.get("due_at") or item.get("due_date"))
    if due is None:
        return False
    current = now or datetime.now(timezone.utc)
    if due.tzinfo is None and current.tzinfo is not None:
        current = current.replace(tzinfo=None)
    return due < current
```

## 14.2 Lab 2 solution - test suite

A strong test suite covers normal behavior, aliases, pass-through behavior, date parsing, and exception behavior. The executable version is in `work_items_api_solution/tests/test_legacy_work_item_logic.py`.

```python
from __future__ import annotations

from datetime import datetime

import pytest

from lab2_assets.legacy_work_item_logic import is_overdue, normalize_status


def test_normalize_status_aliases():
    assert normalize_status("new") == "todo"
    assert normalize_status("WIP") == "in_progress"
    assert normalize_status("completed") == "done"


def test_normalize_status_passes_unknown_values_through():
    assert normalize_status("Blocked") == "blocked"


def test_missing_due_date_is_not_overdue():
    assert is_overdue({"status": "todo"}, now=datetime(2026, 1, 2)) is False


def test_unfinished_past_due_item_is_overdue():
    item = {"status": "todo", "due_date": "2026-01-01"}

    assert is_overdue(item, now=datetime(2026, 1, 2)) is True


def test_done_item_is_not_overdue_even_when_due_date_is_past():
    item = {"status": "done", "due_date": "2026-01-01"}

    assert is_overdue(item, now=datetime(2026, 1, 2)) is False


def test_date_only_strings_are_interpreted_at_midnight():
    item = {"status": "todo", "due_date": "2026-01-02"}

    assert is_overdue(item, now=datetime(2026, 1, 2, 1, 0, 0)) is True


def test_malformed_date_raises_value_error():
    with pytest.raises(ValueError):
        is_overdue({"status": "todo", "due_date": "not-a-date"}, now=datetime(2026, 1, 2))
```

### Why these tests are strong

| Test | What it proves |
|---|---|
| `test_normalize_status_aliases` | Common partner labels map to canonical statuses. |
| `test_normalize_status_passes_unknown_values_through` | Unknown statuses are normalized but not rejected or forced into a known enum. |
| `test_missing_due_date_is_not_overdue` | Absence of due date is safe and returns `False`. |
| `test_unfinished_past_due_item_is_overdue` | Core overdue behavior works for unfinished items. |
| `test_done_item_is_not_overdue_even_when_due_date_is_past` | Completed work is never overdue. |
| `test_date_only_strings_are_interpreted_at_midnight` | Date-only ISO strings are accepted and interpreted consistently. |
| `test_malformed_date_raises_value_error` | Current error behavior is documented and protected. |

### Optional extra tests for advanced cohorts

These are useful but not required in the core lab:

```python
import pytest
from datetime import datetime, timezone

from lab2_assets.legacy_work_item_logic import is_overdue, normalize_status


def test_none_status_defaults_to_todo():
    assert normalize_status(None) == "todo"


def test_unsupported_due_type_raises_type_error():
    with pytest.raises(TypeError):
        is_overdue({"status": "todo", "due_date": 123}, now=datetime(2026, 1, 2))


def test_timezone_aware_now_can_compare_to_naive_due_date():
    item = {"status": "todo", "due_date": "2026-01-01T12:00:00"}
    assert is_overdue(item, now=datetime(2026, 1, 2, tzinfo=timezone.utc)) is True
```

The optional tests help advanced students see how AI can reveal hidden design questions without forcing extra scope into the required lab.

## 14.3 Lab 2 solution - documentation note, AI misses, and optional refactor plan

### Model documentation note

```text
normalize_status(raw) converts common partner status aliases such as "new", "wip", and "completed" into canonical work-item statuses. Unknown statuses are returned after basic trimming, lowercasing, and hyphen-to-underscore normalization.

is_overdue(item, now=None) checks `due_at` or `due_date` and returns True only when the item is not done and the due date is earlier than the comparison time. The function accepts `datetime` values, ISO datetime strings, and date-only strings. Missing due dates return False. Malformed date strings currently raise ValueError, and unsupported due-date value types raise TypeError, so callers should validate user-facing inputs before calling it.
```

### "What AI missed or overclaimed" model answer

```text
The AI explanation was useful for orientation, but it initially overclaimed that date handling was robust. The code accepts common ISO-like strings, but malformed strings raise ValueError and unsupported value types raise TypeError. It also silently passes unknown statuses through, which may be intentional partner compatibility or may be a validation gap. Before refactoring, I would keep these behaviors protected by tests and ask the product/team whether unknown statuses should be accepted or rejected.
```

### Optional / skim if needed: behavior-preserving refactor plan

This refactor discussion is optional. It should not happen until tests pass.

A safe AI prompt:

```text
Do not implement yet. Propose a behavior-preserving refactor plan for legacy_work_item_logic.py. Keep public behavior unchanged, including exception behavior. Identify tests that protect each behavior and list risks before suggesting code changes.
```

A reasonable plan:

1. Keep `normalize_status` public and behavior-compatible.
2. Rename `_parse_due` only if the team agrees it improves readability.
3. Add a small internal helper for timezone normalization only if tests cover naive/aware comparisons.
4. Do not change unknown-status pass-through behavior without a product decision.
5. Do not catch malformed dates unless the API contract is changed.

A bad refactor would silently change malformed-date behavior from "raises" to "returns False" without discussion. That may look safer, but it changes the contract and can hide bad input.

## 15. Privacy and prompt hygiene

Before pasting context into AI, ask:

- Does this contain secrets, tokens, credentials, customer data, private logs, or regulated personal data?
- Can I reproduce the issue with a sanitized example?
- Does company policy allow this tool and this data class?
- Can I ask for a pattern instead of sharing real data?

Sanitized prompt pattern:

```text
Here is a sanitized example with fake IDs and no secrets. Diagnose the pattern and suggest checks I can run locally.
```

## 16. AI-assisted review

Prompt:

```text
Review this diff as a review aid, not as approval.
Identify behavior changes, missing tests, privacy/security concerns, backward compatibility risks, and questions for a human reviewer.
```

Good AI review output is useful because it creates a checklist.

Bad AI review output is dangerous when it sounds like approval.

Human reviewers still own:

- product fit;
- security sign-off;
- architecture fit;
- merge decision;
- production risk.

## [Optional] Terminal and cloud-agent workflows

These are optional in Module 2 because they become more central later.

### Terminal/CLI agent habit

Require command explanation before execution:

```text
Before running any command, explain what it does, what it can modify, expected output, and rollback/inspection options. Wait for approval.
```

### Cloud/coding agent habit

Write issue-shaped tasks:

```text
Include acceptance criteria, out-of-scope files, tests, risks, and PR review checklist. The agent may open a PR but cannot merge or deploy.
```

## 17. [Optional] Homework

See `module2_homework.md` for full instructions and solutions.

Options:

1. Extend the API.
2. Produce a legacy-code onboarding packet.
3. Generate and review an API client.
4. Contribute to a prompt cookbook.

## 18. Takeaways

- Ask for a plan before code.
- Keep changes bounded and reviewable.
- Generated tests are drafts, not proof.
- Existing repos need Map → Bound → Verify.
- API clients, comments, and docs require review for assumptions and overclaims.
- AI review is a review aid, not approval.
- Developers still own correctness, security, and maintainability.
- Legacy-code explanations must separate visible behavior from assumptions.
- Solution material should be revealed only after students attempt the exercise.

<hr/>